# Google Colab T4 Compatible QLoRA Notebook

In [ ]:
!pip install -q transformers==4.51.3 accelerate==1.6.0 peft==0.15.2 datasets==3.5.0 \
             evaluate rouge_score matplotlib numpy scipy
!pip install flash-attn --no-build-isolation -q

## IMPORTANT
After running the install cell, restart the runtime:

Runtime → Restart Session

In [8]:
import torch
import subprocess

print(subprocess.getoutput("nvidia-smi"))

if not torch.cuda.is_available():
    raise RuntimeError("GPU not enabled. Runtime → Change Runtime Type → GPU")

print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")


Wed May 27 13:16:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os

MODEL_ID   = "microsoft/phi-2"
DATASET_ID = "AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.0"
EOS_TOKEN  = "<|endoftext|>"    # Phi-2 EOS token

OUTPUT_DIR  = "./cybersec-qlora"
ADAPTER_DIR = f"{OUTPUT_DIR}/adapter"
RESULTS_DIR = f"{OUTPUT_DIR}/results"

MAX_LENGTH    = 1024
BATCH_SIZE    = 16    # H100 80GB — no quantization, plenty of VRAM
GRAD_ACCUM    = 2     # effective batch = 32
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 3

LORA_R       = 64
LORA_ALPHA   = 128
LORA_DROPOUT = 0.05

SEED = 42

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Config ready.")

In [ ]:
import numpy as np
from datasets import load_dataset, DatasetDict

raw   = load_dataset(DATASET_ID)
full  = raw["train"].filter(lambda ex: bool(ex["user"]) and bool(ex["assistant"]))
# Use full 99k dataset — H100 has no session time constraints
split = full.train_test_split(test_size=0.2, seed=SEED)
ds    = DatasetDict({"train": split["train"], "validation": split["test"]})

print(f"Train      : {len(ds['train'])} examples")
print(f"Validation : {len(ds['validation'])} examples")

a_lens = [len(ex["assistant"].split()) for ex in ds["train"]]
print(f"Answer words — min:{min(a_lens)} max:{max(a_lens)} mean:{np.mean(a_lens):.0f}")

In [ ]:
SYSTEM_PROMPT = (
    "You are a highly specialized AI assistant for advanced cyber-defense. "
    "Deliver accurate, in-depth, actionable guidance on information-security "
    "principles including confidentiality, integrity, availability, "
    "authenticity, non-repudiation, and privacy."
)

def format_prompt(question, answer=None, system=None):
    sys_text = system if system else SYSTEM_PROMPT
    prompt = f"### System:\n{sys_text}\n\n### Question:\n{question}\n\n### Answer:\n"
    if answer is not None:
        prompt += answer + EOS_TOKEN
    return prompt

def make_texts(example):
    return {
        "full_text":   format_prompt(example["user"], example["assistant"], example.get("system")),
        "prefix_text": format_prompt(example["user"], answer=None,          system=example.get("system")),
    }

ds = ds.map(make_texts, desc="Formatting")
print("Sample preview:")
print(ds["train"][0]["full_text"][:400])

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"
print(f"Vocab: {tokenizer.vocab_size:,} | Pad: {tokenizer.pad_token}")

def tokenize(example):
    full_enc   = tokenizer(example["full_text"],   truncation=True,  max_length=MAX_LENGTH, padding=False)
    prefix_enc = tokenizer(example["prefix_text"], truncation=False, padding=False)

    input_ids  = full_enc["input_ids"]
    labels     = list(input_ids)
    prefix_len = len(prefix_enc["input_ids"])
    for i in range(min(prefix_len, len(labels))):
        labels[i] = -100

    return {
        "input_ids":      input_ids,
        "attention_mask": full_enc["attention_mask"],
        "labels":         labels,
    }

tokenized_ds = ds.map(
    tokenize,
    remove_columns=ds["train"].column_names,
    desc="Tokenizing",
)
print(f"Train sample length: {len(tokenized_ds['train'][0]['input_ids'])}")
print("Tokenization done.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM

# No quantization needed — H100 80GB handles Phi-2 2.7B in bf16 easily
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

mem = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded — GPU memory used: {mem:.2f} GB")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Phi-2 target modules (PhiAttention: q/k/v/dense, PhiMLP: fc1/fc2)
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} ({100*trainable/total:.2f}% of {total:,})")

In [ ]:
import json, time
from transformers import (
    TrainingArguments, Trainer, DataCollatorForSeq2Seq, TrainerCallback,
)

class FileLoggerCallback(TrainerCallback):
    def __init__(self, path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        self.path = path
        open(path, "w").close()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        entry = {"step": state.global_step, "epoch": round(state.epoch or 0, 3)}
        entry.update({k: round(v, 6) if isinstance(v, float) else v for k, v in logs.items()})
        with open(self.path, "a") as f:
            f.write(json.dumps(entry) + "\n")

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=False,
    bf16=True,                       # H100 native bf16
    gradient_checkpointing=True,
    logging_strategy="steps",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
    optim="adamw_torch_fused",       # fastest optimizer on H100
    label_names=["labels"],
)

collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors="pt",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=collator,
    callbacks=[FileLoggerCallback(f"{RESULTS_DIR}/training_logs.txt")],
)

print("Starting training...")
t0 = time.perf_counter()
trainer.train()
elapsed = time.perf_counter() - t0
h, m = divmod(int(elapsed), 3600)
print(f"Done in {h}h {m//60}m {m%60}s")

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved → {ADAPTER_DIR}/")

summary = {
    "model_id":        MODEL_ID,
    "dataset_id":      DATASET_ID,
    "quantization":    "none (bf16 full precision)",
    "epochs":          NUM_EPOCHS,
    "batch_size":      BATCH_SIZE,
    "effective_batch": BATCH_SIZE * GRAD_ACCUM,
    "learning_rate":   LEARNING_RATE,
    "max_length":      MAX_LENGTH,
    "lora_r":          LORA_R,
    "lora_alpha":      LORA_ALPHA,
    "train_examples":  len(tokenized_ds["train"]),
    "val_examples":    len(tokenized_ds["validation"]),
    "wall_time_sec":   round(elapsed),
}
with open(f"{RESULTS_DIR}/training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Summary saved.")

In [ ]:
import evaluate as ev

rouge = ev.load("rouge")
model.eval()

val_raw = ds["validation"].select(range(min(10, len(ds["validation"]))))
preds, refs, lines = [], [], []

for i, ex in enumerate(val_raw):
    prompt = format_prompt(ex["user"], answer=None, system=ex.get("system"))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    preds.append(decoded)
    refs.append(ex["assistant"])
    lines += [
        f"=== SAMPLE {i+1} ===",
        f"Q: {ex['user'][:120]}",
        f"PRED: {decoded[:300]}",
        f"REF : {ex['assistant'][:300]}",
        "",
    ]

scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
print("ROUGE scores:")
for k, v in scores.items():
    print(f"  {k}: {v:.4f}")

with open(f"{RESULTS_DIR}/sample_predictions.txt", "w") as f:
    f.write("\n".join(lines))
scores["num_samples"] = len(preds)
with open(f"{RESULTS_DIR}/metrics.json", "w") as f:
    json.dump({k: round(v, 4) if isinstance(v, float) else v for k, v in scores.items()}, f, indent=2)
print(f"Saved metrics.json and sample_predictions.txt → {RESULTS_DIR}/")

In [ ]:
import matplotlib.pyplot as plt

logs = []
with open(f"{RESULTS_DIR}/training_logs.txt") as f:
    for line in f:
        try:
            logs.append(json.loads(line.strip()))
        except:
            pass

train_steps = [l["step"] for l in logs if "loss" in l and "eval_loss" not in l]
train_loss  = [l["loss"]  for l in logs if "loss" in l and "eval_loss" not in l]
eval_steps  = [l["step"]  for l in logs if "eval_loss" in l]
eval_loss   = [l["eval_loss"] for l in logs if "eval_loss" in l]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_loss, label="Train loss", alpha=0.8)
if eval_steps:
    ax.plot(eval_steps, eval_loss, label="Val loss", linewidth=2)
ax.set_xlabel("Step"); ax.set_ylabel("Loss")
ax.set_title("Training vs Validation Loss — Cybersecurity QA LoRA")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/learning_curve.png", dpi=150)
plt.show()
print(f"Saved → {RESULTS_DIR}/learning_curve.png")

In [ ]:
import shutil

shutil.make_archive("cybersec_results", "zip", ".", f"{OUTPUT_DIR}/results")
shutil.make_archive("cybersec_adapter", "zip", ".", f"{OUTPUT_DIR}/adapter")

print("Zipped → cybersec_results.zip and cybersec_adapter.zip")
print("Download via your cloud provider's file manager or:")
print("  scp user@<host>:~/cybersec_results.zip .")
print("  scp user@<host>:~/cybersec_adapter.zip .")